# Correlation-based feature selection (Fold 0)

In [ ]:
import pickle
import numpy as np
import pandas as pd
from scipy.stats import pointbiserialr
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns



In [ ]:
with open ("../data/top_features/mimic_top100_features.pkl", "rb") as f:
    mimic_top_features = pickle.load(f)
    
mimic_top100 = set(int(x) for x in mimic_top_features)


In [ ]:
# We have three constraints for the correlation based feature selection
#1. there should be a minumum count of features to apply correlation so basically we keep iteids that are measures at least 10 times across admissions
#2. for each itemid there should be positive and negative labels so we are ignoring itemids only measured in postive case epecially
#3. if the value is consistant that itemids is removed because computing corr needs variance in data

#For example in cohort F35-A41 123 itemids have been removed

#TODO: check later 

# Too few features are selected 21 among 224

In [ ]:
import sys
sys.path.append(str(Path.cwd().parent))
from config.constants import PROJECT_ROOT, MIMIC_IV_PATH

ROOT = PROJECT_ROOT
CORR_DIR = ROOT / "saved_data" / "features_selected_corr"
METHODS = {"fdr": CORR_DIR/"fdr", "top100": CORR_DIR / "top100", "mrmr100": CORR_DIR / "mrmr100"}

d_labitems = pd.read_csv(Path(MIMIC_IV_PATH)/"hosp"/"d_labitems.csv.gz")



cohort_selected = {}
for method, base in METHODS.items():
    for cohort_dir in sorted(p for p in base.iterdir() if p.is_dir()):
        pkl_path = cohort_dir / "fold_0" / "selected_itemids.pkl"
        if not pkl_path.exists():
            continue
        with open(pkl_path, "rb") as f:
            cohort_selected[(method, cohort_dir.name)] = set(pickle.load(f))
sel = pd.DataFrame(
    [(method, cohort, itemid) for (method, cohort), itemids in cohort_selected.items() for itemid in itemids],
    columns=["method", "cohort", "itemid"],
)
sel["in_mimic_top100"] = sel["itemid"].isin(mimic_top100)

In [ ]:
overlap_df = (sel.groupby(["method", "cohort"]).agg(n_selected=("itemid", "size"), n_in_top100=("in_mimic_top100", "sum")).reset_index())
overlap_df["n_new"] = overlap_df["n_selected"] - overlap_df["n_in_top100"]

cohort_order = (overlap_df.groupby("cohort")["n_selected"].sum().sort_values(ascending=False).index)

method_titles = {"fdr": "corr_fdr", "top100": "corr_top100", "mrmr100": "MRMR"}

fig, axes = plt.subplots(len(METHODS), 1, figsize=(14, 10))
for ax, method in zip(axes, METHODS):
    d = (overlap_df[overlap_df["method"] == method]
         .set_index("cohort").reindex(cohort_order).reset_index())
    ax.bar(d["cohort"], d["n_in_top100"], label="in MIMIC top100")
    ax.bar(d["cohort"], d["n_new"], bottom=d["n_in_top100"], label="new / not in top100")
    ax.set_xticks(range(len(d)))
    ax.set_xticklabels(d["cohort"], rotation=90, fontsize=8)
    ax.tick_params(axis="y", labelsize=12)
    ax.set_ylabel("# selected features", fontsize=14)
    ax.set_title(method_titles.get(method, method), fontsize=16)
    ax.legend()
plt.tight_layout()
plt.show()

plot for a selected cohort

In [ ]:
cohort_name = "J38-J06"
method = "top100"
df = sel[(sel.cohort == cohort_name) & (sel.method == method)].merge(d_labitems[["itemid", "label"]], on="itemid")
df = df.sort_values(["in_mimic_top100", "label"])

df["y"] = df.groupby("in_mimic_top100").cumcount()
df["x"] = df["in_mimic_top100"].map({False: 0, True: 1})
colors = df["in_mimic_top100"].map({True: "seagreen", False: "salmon"})

plt.figure(figsize=(8, max(4, df["y"].max() * 0.3)))
plt.scatter(df["x"], df["y"], color=colors, s=60, zorder=3)
for _, row in df.iterrows():
    plt.text(row["x"] + 0.05, row["y"], row["label"], va="center", ha="left", fontsize=8)

plt.xticks([0, 1], ["not in MIMIC top100", "in MIMIC top100"])
plt.xlim(-0.5, 2.2)
plt.gca().invert_yaxis()
plt.yticks([])
plt.title(f"{cohort_name} ({method}) selected features")
plt.tight_layout()
plt.show()

#Per-cohort feature overlap across selection methods (dot plot)

In [ ]:
cohort_name = "K59-J69"
methods_to_compare = ["mrmr100", "top100", "fdr"]
method_labels = ["MRMR", "Corr-top100", "Corr-FDR"]

cd = (sel[(sel.cohort == cohort_name) & (sel.method.isin(methods_to_compare))]
      .merge(d_labitems[["itemid", "label"]], on="itemid", how="left"))

features = (cd[["itemid", "label", "in_mimic_top100"]]
            .drop_duplicates()
            .sort_values(["in_mimic_top100", "label"], ascending=[False, True])
            .reset_index(drop=True))

half = (len(features) + 1) // 2
groups = [features.iloc[:half], features.iloc[half:]]
max_rows = max(len(g) for g in groups)

xpos = {m: i for i, m in enumerate(methods_to_compare)}

fig, axes = plt.subplots(1, 2, figsize=(14, max(4, max_rows * 0.28)))
for ax, g in zip(axes, groups):
    ypos = dict(zip(g["itemid"], range(len(g))))
    sub = cd[cd["itemid"].isin(g["itemid"])]
    for _, row in sub.iterrows():
        color = "seagreen" if row["in_mimic_top100"] else "salmon"
        ax.scatter(xpos[row["method"]], ypos[row["itemid"]], color=color, s=70, zorder=3)
    for _, row in g.iterrows():
        ax.text(len(methods_to_compare) - 0.3, ypos[row["itemid"]], row["label"],
                va="center", ha="left", fontsize=8)
    ax.set_xticks(range(len(methods_to_compare)))
    ax.set_xticklabels(method_labels)
    ax.set_xlim(-0.5, len(methods_to_compare) + 2)
    ax.set_ylim(-1, max_rows)
    ax.invert_yaxis()
    ax.set_yticks([])

from matplotlib.lines import Line2D
fig.legend(handles=[
    Line2D([0], [0], marker="o", color="w", markerfacecolor="seagreen", markersize=8, label="in MIMIC top100"),
    Line2D([0], [0], marker="o", color="w", markerfacecolor="salmon", markersize=8, label="not in MIMIC top100"),
], loc="upper right")
#fig.suptitle(f"{cohort_name}: Features selected per method")
plt.tight_layout()
plt.show()


## Compare features selection methods training results

In [ ]:
import pandas as pd
from pathlib import Path

results_root = Path("../saved_data/results")
cohort_list = [l.strip() for l in open("../saved_data/cohorts/DTB/representative_cohorts.txt").readlines()[1:] if l.strip()]

def load_result(cohort, prefix, fold=0):
    path = (results_root / cohort / "random_forest" / prefix / f"fold_{fold}"
            / "agg_int_24" / "impute_fill" / "variant_VMD" / "results_final.csv")
    if not path.exists():
        return None
    df = pd.read_csv(path, on_bad_lines="skip")
    test_rows = df[df["split"] == "test"]
    if test_rows.empty:
        return None
    row = test_rows.iloc[-1]  # last/final run for that split
    return {"auroc": row["auroc"], "auprc": row["auprc"], "f1": row["f1"]}

rows = []
for cohort in cohort_list:
    # old run: prefer the 030626 rerun where it exists, else fall back to the original 200526/190526 run
    old_prefix = next(
        (c for c in ["030626", "200526", "190526"]
         if (results_root / cohort / "random_forest" / c / "fold_0").exists()),
        None,
    )

    entry = {"cohort": cohort}
    for label, prefix in [("old", old_prefix), ("top100", "corr_feat_training"), ("mrmr", "mrmr_feat_training")]:
        res = load_result(cohort, prefix) if prefix else None
        entry[f"{label}_auroc"] = res["auroc"] if res else None
        entry[f"{label}_auprc"] = res["auprc"] if res else None
        entry[f"{label}_f1"] = res["f1"] if res else None
    rows.append(entry)

results_df = pd.DataFrame(rows)
#print("Missing values per column:")
#print(results_df.isna().sum())
#results_df
results_df= results_df.dropna(subset=["old_auroc"])


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

variants = ["old_auroc", "top100_auroc", "mrmr_auroc"]
labels = ["mimic-top100", "corr-top100", "MRMR"]
colors = ["#2a78d6", "#008300", "#e87ba4"]

data = [results_df[v].dropna().values for v in variants]

fig, ax = plt.subplots(figsize=(7, 5))

bp = ax.boxplot(
    data,
    labels=labels,
    widths=0.5,
    patch_artist=True,
    showfliers=False,
    medianprops=dict(color="black", linewidth=2),
)

for patch, color in zip(bp["boxes"], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.25)
    patch.set_edgecolor(color)
    patch.set_linewidth(2)

for i, (vals, color) in enumerate(zip(data, colors), start=1):
    jitter = np.random.normal(loc=i, scale=0.06, size=len(vals))
    ax.scatter(jitter, vals, color=color, alpha=0.6, s=18, zorder=3)

ax.set_ylabel("AUROC")
ax.set_title("Random Forest AUROC by feature selection (100 cohorts, fold 0)")
ax.grid(axis="y", linestyle="--", alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
from scipy import stats
from statsmodels.stats.multitest import multipletests
import pandas as pd

pairs = [
    ("mrmr_auroc", "top100_auroc"),
    ("top100_auroc", "old_auroc"),
    ("mrmr_auroc", "old_auroc"),
]

rows = []
for a, b in pairs:
    sub = results_df[[a, b]].dropna()
    t_stat, t_p = stats.ttest_rel(sub[a], sub[b])
    w_stat, w_p = stats.wilcoxon(sub[a], sub[b])
    rows.append({
        "comparison": f"{a} vs {b}",
        "n": len(sub),
        "mean_diff": (sub[a] - sub[b]).mean(),
        "ttest_p": t_p,
        "wilcoxon_p": w_p,
    })

sig_df = pd.DataFrame(rows)

# Holm-Bonferroni correction across the 3 comparisons, for each test separately
for col in ["ttest_p", "wilcoxon_p"]:
    reject, p_corrected, _, _ = multipletests(sig_df[col], method="holm")
    sig_df[f"{col}_corrected"] = p_corrected
    sig_df[f"{col}_significant"] = reject

sig_df


In [ ]:
results_df.sort_values(by= "top100_auroc",ascending=False)

In [ ]:
results_df[results_df["cohort"].str.startswith("J")]


In [ ]:
import pandas as pd
from pathlib import Path
import re

mimic_d_items =  pd.read_csv(Path(MIMIC_IV_PATH)/"hosp"/"d_labitems.csv.gz")


cohort = "J38-J06"  # set the cohort you want

prefixes = {
    "old": next((p for p in ["030626", "200526", "190526"]
                 if (Path("../saved_data/results")/cohort/"random_forest"/p/"fold_0").exists()), None),
    "top100": "corr_feat_training",
    "mrmr": "mrmr_feat_training",
}

top10_by_variant = {}
for label, prefix in prefixes.items():
    if prefix is None:
        continue
    fi_path = (Path("../saved_data/results") / cohort / "random_forest" / prefix
               / "fold_0" / "agg_int_24" / "impute_fill" / "variant_VMD" / "feature_importances.csv")
    fi = pd.read_csv(fi_path)
    top10_by_variant[label] = fi.sort_values("importance", ascending=False).head(10).reset_index(drop=True)

'''for label, df in top10_by_variant.items():
    print(f"\n=== {label} ({cohort}) ===")
    print(df)'''


def label_features(df):
    df = df.copy()
    parsed = df["feature"].str.extract(r"^(\d+)(_.*)?$")
    df["itemid"] = pd.to_numeric(parsed[0], errors="coerce").astype("Int64")
    df["suffix"] = parsed[1].fillna("")
    df = df.merge(mimic_d_items[["itemid", "label"]], on="itemid", how="left")
    return df

top10_labeled = {label: label_features(df) for label, df in top10_by_variant.items()}



In [ ]:
comparison = pd.DataFrame({
    label: (df["label"].fillna(df["feature"]) + " (" + df["feature"] + ")").values
    for label, df in top10_labeled.items()
})
comparison = comparison.rename(columns={
    "old": "mimic-top100",
    "top100": "corr-top100",
    "mrmr": "MRMR",
})
comparison

comparison.index = range(1, 11)
comparison.index.name = "rank"
comparison


In [ ]:
import pickle
with open ("../data/top_features/mimic_top100_features.pkl", "rb") as f:
    mimic_top_features = pickle.load(f)

In [ ]:
check_ids = ["50815","50804"]
{i: (i in mimic_top_features) for i in check_ids}


In [ ]:
sub_mrmr = results_df[["cohort", "mrmr_auroc", "old_auroc"]].dropna()
sub_top100 = results_df[["cohort", "top100_auroc", "old_auroc"]].dropna()

mrmr_better = set(sub_mrmr.loc[sub_mrmr["mrmr_auroc"] > sub_mrmr["old_auroc"], "cohort"])
top100_better = set(sub_top100.loc[sub_top100["top100_auroc"] > sub_top100["old_auroc"], "cohort"])

print(f"mrmr better than old: {len(mrmr_better)} cohorts")
print(f"top100 better than old: {len(top100_better)} cohorts")
print(f"overlap (both better): {len(mrmr_better & top100_better)}")
print(f"only mrmr better: {len(mrmr_better - top100_better)}")
print(f"only top100 better: {len(top100_better - mrmr_better)}")
print(f"neither better: {len(set(results_df['cohort']) - mrmr_better - top100_better)}")

results_df[results_df["cohort"].isin(mrmr_better - top100_better)][["cohort", "old_auroc", "top100_auroc", "mrmr_auroc"]]
